In [1]:
import json
import random
import re
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from torch import nn
from torch.nn.utils.rnn import pack_padded_sequence
from torch.utils.data import DataLoader, Dataset
from transformers import BertTokenizer

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Set up Hugging Face cache directory
cache_dir = Path(r"C:/Users/User/Documents/devanasokan_fyp/huggingface_cache")
cache_dir.mkdir(parents=True, exist_ok=True)
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased", cache_dir=str(cache_dir))
print(f"Tokenizer vocab size: {len(tokenizer)}")

c:\Users\User\Documents\devanasokan_fyp\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda
Tokenizer vocab size: 30522


In [2]:
data_path = Path(r"C:/Users/User/Documents/devanasokan_fyp/preparation/modeldata.csv")
df = pd.read_csv(data_path)
df = df[["verse", "label"]].dropna().drop_duplicates().reset_index(drop=True)

print(df.shape)
print(df["label"].value_counts().sort_index())

(22878, 2)
label
0    11439
1    11439
Name: count, dtype: int64


#### Text Cleaning and Train/Validation/Test Split

The LSTM should only see text from the training split when building the vocabulary. That avoids leaking information from validation or test lyrics into the tokenizer.

In [3]:
def clean_text(text: str) -> str:
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s']", " ", text) # Replace non-alphanumeric characters with spaces
    text = re.sub(r"\s+", " ", text).strip() # Replace multiple spaces with a single space and trim
    return text

train_df, temp_df = train_test_split(
    df,
    test_size=0.2,
    random_state=seed,
    stratify=df["label"],
)
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=seed,
    stratify=temp_df["label"],
)

for frame in (train_df, val_df, test_df):
    frame.loc[:, "clean_verse"] = frame["verse"].apply(clean_text)

print("train:", train_df.shape, train_df["label"].value_counts().sort_index().to_dict())
print("val:", val_df.shape, val_df["label"].value_counts().sort_index().to_dict())
print("test:", test_df.shape, test_df["label"].value_counts().sort_index().to_dict())

train: (18302, 3) {0: 9151, 1: 9151}
val: (2288, 3) {0: 1144, 1: 1144}
test: (2288, 3) {0: 1144, 1: 1144}


In [4]:
MAX_LEN = 180


def tokenize(text: str) -> list[str]:
    return tokenizer.tokenize(text)


counter = Counter()
for text in train_df["clean_verse"]:
    counter.update(tokenize(text))

vocab = tokenizer.get_vocab()

print(f"Vocabulary size: {len(vocab)}")
print("Most common tokens:", counter.most_common(10))

Token indices sequence length is longer than the specified maximum sequence length for this model (544 > 512). Running this sequence through the model will result in indexing errors


Vocabulary size: 30522
Most common tokens: [('i', 54906), ('you', 40629), ('the', 31056), ('to', 24273), ('it', 20227), ('a', 19852), ('me', 18337), ('and', 17799), ('not', 15771), ('is', 15142)]


In [5]:
def numericalize(text: str) -> tuple[torch.Tensor, torch.Tensor]:
    token_ids = tokenizer.encode(
        text,
        add_special_tokens=False,
        truncation=True,
        max_length=MAX_LEN,
    )
    if not token_ids:
        token_ids = [tokenizer.unk_token_id]
    length = len(token_ids)
    if len(token_ids) < MAX_LEN:
        token_ids += [tokenizer.pad_token_id] * (MAX_LEN - len(token_ids))
    return torch.tensor(token_ids, dtype=torch.long), torch.tensor(length, dtype=torch.long)

sample_ids, sample_length = numericalize(train_df.iloc[0]["clean_verse"] )
print("sample length:", sample_length.item())
print("sample ids:", sample_ids[:20].tolist())

sample length: 46
sample ids: [2026, 2026, 2092, 2009, 2003, 25085, 2066, 2057, 2024, 2035, 2183, 2000, 3280, 2035, 2183, 2000, 3280, 9061, 9061, 2524]


In [6]:
class LyricsDataset(Dataset):
    def __init__(self, frame: pd.DataFrame):
        self.texts = frame["clean_verse"].tolist()
        self.labels = frame["label"].astype(np.float32).tolist()

    def __len__(self) -> int:
        return len(self.texts)

    def __getitem__(self, index: int):
        input_ids, length = numericalize(self.texts[index])
        label = torch.tensor(self.labels[index], dtype=torch.float32)
        return input_ids, length, label

## Hyperparameter Tuning with Optuna

We'll search for the best hyperparameters using Optuna, which efficiently explores different combinations and focuses on promising regions. We'll tune:
- Learning rate (1e-4 to 1e-2)
- Embedding dimension (64, 128, 256)
- Hidden dimension (128, 256, 512)
- Number of layers (1, 2, 3)
- Dropout rate (0.1 to 0.5)
- Batch size (16, 32, 64)

In [7]:
class LSTMClassifier(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        embedding_dim: int = 128,
        hidden_dim: int = 128,
        num_layers: int = 2,
        bidirectional: bool = True,
        dropout: float = 0.3,
    ):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=tokenizer.pad_token_id)
        self.lstm = nn.LSTM(
            embedding_dim,
            hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
            bidirectional=bidirectional,
        )
        output_dim = hidden_dim * 2 if bidirectional else hidden_dim
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(output_dim, 1)

    def forward(self, input_ids: torch.Tensor, lengths: torch.Tensor) -> torch.Tensor:
        embedded = self.embedding(input_ids)
        packed = pack_padded_sequence(
            embedded,
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=False,
        )
        _, (hidden, _) = self.lstm(packed)
        if self.lstm.bidirectional:
            hidden = torch.cat((hidden[-2], hidden[-1]), dim=1)
        else:
            hidden = hidden[-1]
        logits = self.fc(self.dropout(hidden))
        return logits.squeeze(1)

In [8]:
def run_epoch(model, loader: DataLoader, criterion, optimizer=None, training: bool = True):
    model.train() if training else model.eval()

    total_loss = 0.0
    all_predictions = []
    all_targets = []

    for input_ids, lengths, labels in loader:
        input_ids = input_ids.to(device)
        lengths = lengths.to(device)
        labels = labels.to(device)

        with torch.set_grad_enabled(training):
            logits = model(input_ids, lengths)
            loss = criterion(logits, labels)

            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

        predictions = (torch.sigmoid(logits) >= 0.5).long()
        all_predictions.extend(predictions.detach().cpu().tolist())
        all_targets.extend(labels.detach().cpu().long().tolist())
        total_loss += loss.item() * input_ids.size(0)

    average_loss = total_loss / len(loader.dataset)
    accuracy = accuracy_score(all_targets, all_predictions)
    return average_loss, accuracy

In [ ]:
import optuna
from optuna.pruners import MedianPruner


def objective(trial: optuna.Trial):
    # Suggest hyperparameters
    learning_rate = trial.suggest_float("learning_rate", 1e-4, 1e-2, log=True)
    embedding_dim = trial.suggest_categorical("embedding_dim", [64, 128, 256])
    hidden_dim = trial.suggest_categorical("hidden_dim", [128, 256, 512])
    num_layers = trial.suggest_int("num_layers", 1, 3)
    dropout = trial.suggest_float("dropout", 0.1, 0.5)
    batch_size = trial.suggest_categorical("batch_size", [16, 32, 64])

    # Create data loaders with the suggested batch size
    train_loader = DataLoader(LyricsDataset(train_df), batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(LyricsDataset(val_df), batch_size=batch_size, shuffle=False)

    # Initialize model with suggested hyperparameters
    model = LSTMClassifier(
        vocab_size=len(tokenizer),
        embedding_dim=embedding_dim,
        hidden_dim=hidden_dim,
        num_layers=num_layers,
        dropout=dropout,
    ).to(device)

    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

    # Train for a few epochs
    num_epochs = 5
    best_val_loss = float("inf")

    for epoch in range(num_epochs):
        train_loss, train_acc = run_epoch(model, train_loader, criterion, optimizer, training=True)
        val_loss, val_acc = run_epoch(model, val_loader, criterion, training=False)

        print(f"  Trial {trial.number}, Epoch {epoch + 1}/{num_epochs}: val_loss={val_loss:.4f}, val_acc={val_acc:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss

        # Prune trial if validation loss is not improving
        trial.report(val_loss, epoch)
        if trial.should_prune():
            raise optuna.TrialPruned()

    return best_val_loss

In [10]:
# Create a study and run the hyperparameter search
study = optuna.create_study(
    direction="minimize",  # Minimize validation loss
    pruner=MedianPruner(),
)

study.optimize(objective, n_trials=6, show_progress_bar=True)

[I 2026-07-01 23:30:45,174] A new study created in memory with name: no-name-181d0651-2c36-48ba-8173-dfe61a6d8123
  0%|          | 0/6 [00:00<?, ?it/s]

  Trial 0, Epoch 1/3: val_loss=0.4269, val_acc=0.8011
  Trial 0, Epoch 2/3: val_loss=0.3912, val_acc=0.8335


Best trial: 0. Best value: 0.391153:  17%|█▋        | 1/6 [02:00<10:02, 120.58s/it]

  Trial 0, Epoch 3/3: val_loss=0.4265, val_acc=0.8204
[I 2026-07-01 23:32:45,744] Trial 0 finished with value: 0.39115278787546226 and parameters: {'learning_rate': 0.003869444703900451, 'embedding_dim': 256, 'hidden_dim': 256, 'num_layers': 1, 'dropout': 0.1286898923047208, 'batch_size': 32}. Best is trial 0 with value: 0.39115278787546226.
  Trial 1, Epoch 1/3: val_loss=0.6101, val_acc=0.6827
  Trial 1, Epoch 2/3: val_loss=0.6181, val_acc=0.6814


Best trial: 0. Best value: 0.391153:  33%|███▎      | 2/6 [05:01<10:24, 156.10s/it]

...

  Trial 3, Epoch 3/3: val_loss=0.5433, val_acc=0.7417
[I 2026-07-01 23:42:51,214] Trial 3 finished with value: 0.5432937933014823 and parameters: {'learning_rate': 0.00010229150079869277, 'embedding_dim': 128, 'hidden_dim': 256, 'num_layers': 2, 'dropout': 0.3785245995189338, 'batch_size': 16}. Best is trial 0 with value: 0.39115278787546226.
  Trial 4, Epoch 1/3: val_loss=0.5869, val_acc=0.6740
  Trial 4, Epoch 2/3: val_loss=0.4821, val_acc=0.7583


Best trial: 0. Best value: 0.391153:  83%|████████▎ | 5/6 [18:36<04:31, 271.02s/it]

  Trial 4, Epoch 3/3: val_loss=0.4383, val_acc=0.7863
[I 2026-07-01 23:49:21,736] Trial 4 finished with value: 0.43826244375505646 and parameters: {'learning_rate': 0.0005243569134930894, 'embedding_dim': 64, 'hidden_dim': 256, 'num_layers': 2, 'dropout': 0.3767558745555626, 'batch_size': 16}. Best is trial 0 with value: 0.39115278787546226.


Best trial: 0. Best value: 0.391153: 100%|██████████| 6/6 [20:47<00:00, 207.99s/it]

  Trial 5, Epoch 1/3: val_loss=0.7145, val_acc=0.4996
[I 2026-07-01 23:51:33,114] Trial 5 pruned. 


In [11]:
# Print best trial results
best_trial = study.best_trial

print(f"Best validation loss: {best_trial.value:.4f}")
print("Best hyperparameters:")
for key, value in best_trial.params.items():
    print(f"  {key}: {value}")

Best validation loss: 0.3912
Best hyperparameters:
  learning_rate: 0.003869444703900451
  embedding_dim: 256
  hidden_dim: 256
  num_layers: 1
  dropout: 0.1286898923047208
  batch_size: 32


## Final Training with Best Hyperparameters

Train the final model using the best hyperparameters found by Optuna on the full train set, and evaluate on the test set.

In [12]:
# Extract best hyperparameters
best_params = best_trial.params
best_learning_rate = best_params["learning_rate"]
best_embedding_dim = best_params["embedding_dim"]
best_hidden_dim = best_params["hidden_dim"]
best_num_layers = best_params["num_layers"]
best_dropout = best_params["dropout"]
best_batch_size = best_params["batch_size"]

# Create data loaders with best batch size
train_loader = DataLoader(LyricsDataset(train_df), batch_size=best_batch_size, shuffle=True)
val_loader = DataLoader(LyricsDataset(val_df), batch_size=best_batch_size, shuffle=False)
test_loader = DataLoader(LyricsDataset(test_df), batch_size=best_batch_size, shuffle=False)

# Initialize final model with best hyperparameters
model = LSTMClassifier(
    vocab_size=len(tokenizer),
    embedding_dim=best_embedding_dim,
    hidden_dim=best_hidden_dim,
    num_layers=best_num_layers,
    dropout=best_dropout,
).to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=best_learning_rate)

print("Final model architecture:")
print(model)

Final model architecture:
LSTMClassifier(
  (embedding): Embedding(30522, 256, padding_idx=0)
  (lstm): LSTM(256, 256, batch_first=True, bidirectional=True)
  (dropout): Dropout(p=0.1286898923047208, inplace=False)
  (fc): Linear(in_features=512, out_features=1, bias=True)
)


In [13]:
num_epochs = 10
best_val_loss = float("inf")
best_model_path = Path("lstm_lyrics_classifier.pt")
history = []

for epoch in range(1, num_epochs + 1):
    train_loss, train_acc = run_epoch(model, train_loader, criterion, optimizer, training=True)
    val_loss, val_acc = run_epoch(model, val_loader, criterion, training=False)

    history.append(
        {
            "epoch": epoch,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "val_loss": val_loss,
            "val_acc": val_acc,
        }
    )

    print(
        f"Epoch {epoch:02d} | "
        f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} | "
        f"val_loss={val_loss:.4f} val_acc={val_acc:.4f}"
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), best_model_path)

history_df = pd.DataFrame(history)
history_df

Epoch 01 | train_loss=0.5912 train_acc=0.6800 | val_loss=0.5122 val_acc=0.7627
Epoch 02 | train_loss=0.3950 train_acc=0.8232 | val_loss=0.4334 val_acc=0.8029
Epoch 03 | train_loss=0.2530 train_acc=0.8968 | val_loss=0.4512 val_acc=0.8252
Epoch 04 | train_loss=0.1505 train_acc=0.9433 | val_loss=0.5436 val_acc=0.8099
Epoch 05 | train_loss=0.1011 train_acc=0.9637 | val_loss=0.5870 val_acc=0.8186
Epoch 06 | train_loss=0.0737 train_acc=0.9749 | val_loss=0.6496 val_acc=0.8239
Epoch 07 | train_loss=0.0619 train_acc=0.9790 | val_loss=0.6847 val_acc=0.8287
Epoch 08 | train_loss=0.0603 train_acc=0.9794 | val_loss=0.7726 val_acc=0.8226
Epoch 09 | train_loss=0.0608 train_acc=0.9797 | val_loss=0.7413 val_acc=0.8230
Epoch 10 | train_loss=0.0509 train_acc=0.9821 | val_loss=0.9473 val_acc=0.8208


,epoch,train_loss,train_acc,val_loss,val_acc
0,1,0.591157,0.680035,0.512211,0.762675
1,2,0.395036,0.823243,0.433422,0.802885
2,3,0.253018,0.896842,0.451192,0.825175
3,4,0.150521,0.943285,0.543639,0.809878
4,5,0.101112,0.963720,0.587019,0.818619
5,6,0.073732,0.974866,0.649586,0.823864
6,7,0.061930,0.978964,0.684704,0.828671
7,8,0.060277,0.979401,0.772576,0.822552
8,9,0.060822,0.979729,0.741308,0.822990
9,10,0.050907,0.982133,0.947297,0.820804


In [14]:
# Load best checkpoint and evaluate on test
model.load_state_dict(torch.load(best_model_path, map_location=device))
test_loss, test_acc = run_epoch(model, test_loader, criterion, training=False)

print(f"\nTest loss: {test_loss:.4f}")
print(f"Test accuracy: {test_acc:.4f}")


Test loss: 0.4381
Test accuracy: 0.7880


In [15]:
artifact_dir = Path("lstm_artifacts")
artifact_dir.mkdir(exist_ok=True)

model_path = artifact_dir / "lyrics_lstm.pt"
vocab_path = artifact_dir / "lyrics_vocab.json"
params_path = artifact_dir / "best_hyperparams.json"

torch.save(model.state_dict(), model_path)
with open(vocab_path, "w", encoding="utf-8") as vocab_file:
    json.dump(vocab, vocab_file, ensure_ascii=False, indent=2)
with open(params_path, "w", encoding="utf-8") as params_file:
    json.dump(best_params, params_file, indent=2)

print("Artifacts saved:")
print(f"  Model: {model_path}")
print(f"  Vocab: {vocab_path}")
print(f"  Best hyperparameters: {params_path}")

Artifacts saved:
  Model: lstm_artifacts\lyrics_lstm.pt
  Vocab: lstm_artifacts\lyrics_vocab.json
  Best hyperparameters: lstm_artifacts\best_hyperparams.json


In [16]:
def predict_text(text: str):
    model.eval()
    cleaned_text = clean_text(text)
    input_ids, length = numericalize(cleaned_text)
    with torch.no_grad():
        logits = model(input_ids.unsqueeze(0).to(device), length.unsqueeze(0).to(device))
        probability = torch.sigmoid(logits).item()
        prediction = int(probability >= 0.5)
    return probability, prediction


sample_probability, sample_prediction = predict_text("you in the dark")
print({"probability": sample_probability, "prediction": sample_prediction})

{'probability': 0.27629590034484863, 'prediction': 0}
